In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

options = Options()
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

driver.get("https://www.google.com")
print(driver.title)
driver.quit()

Google


In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")

try:
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

    driver.get("https://www.google.com")
    print(f"CONECTADO! Título de la página: {driver.title}")

    driver.quit()

except Exception as e:
    print(f"Error: {e}")

CONECTADO! Título de la página: Google


In [4]:
import time
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

# ==========================================================
# 1. MOTOR DE NAVEGACIÓN
# ==========================================================
def iniciar_navegador():
    options = Options()
    options.binary_location = "/usr/bin/google-chrome"
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    )

    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

# ==========================================================
# 2. CONFIGURACIÓN DEL CASO
# ==========================================================
URL_OBJETIVO = "https://www.chileautos.cl/vehiculos/autos-veh%C3%ADculo/"

driver = iniciar_navegador()
dataset_final = []

try:
    print("Conectando a la fuente de datos...")
    driver.get(URL_OBJETIVO)
    time.sleep(6)

    # Tomamos títulos visibles de vehículos
    titulos = driver.find_elements(By.TAG_NAME, "h3")
    print(f"Se encontraron {len(titulos)} títulos potenciales.")

    for item in titulos:
        try:
            nombre = item.text.strip()

            if not nombre:
                continue

            # Buscar el bloque padre cercano para encontrar el precio
            contenedor = item.find_element(By.XPATH, "./ancestor::*[self::article or self::div][1]")
            texto_bloque = contenedor.text

            match_precio = re.search(r"\$\s?[\d\.,]+", texto_bloque)
            if not match_precio:
                continue

            precio = match_precio.group(0)

            dataset_final.append({
                "auto": nombre,
                "precio": precio,
                "grupo": "Automotriz",
                "status": 1.0
            })

        except:
            continue

    # Quitar duplicados
    df = pd.DataFrame(dataset_final).drop_duplicates()

    if not df.empty:
        nombre_archivo = "datos_extraidos_automotriz.csv"
        df.to_csv(nombre_archivo, index=False)
        print(f"Proceso exitoso. Archivo generado: {nombre_archivo}")
        print(df.head())
    else:
        print("No se capturaron datos. Revisar selectores o estructura.")

finally:
    driver.quit()
    print("Navegador cerrado.")

Conectando a la fuente de datos...
Se encontraron 0 títulos potenciales.
No se capturaron datos. Revisar selectores o estructura.
Navegador cerrado.
